In [104]:
import os
import re
import sqlite3
import pandas as pd
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch

In [105]:
llm = ChatGroq(model="openai/gpt-oss-20b",api_key=os.environ["GROQ_API_KEY"])
web_search = TavilySearch(max_results=5)

In [106]:
conn = sqlite3.connect("lab_book.db",check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS tests (
    test_id TEXT PRIMARY KEY,
    test_name TEXT NOT NULL,
    category TEXT NOT NULL,
    description TEXT,
    price REAL NOT NULL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS slots (
    slot_id TEXT PRIMARY KEY,
    test_id TEXT NOT NULL,
    slot_date TEXT NOT NULL,
    slot_time TEXT NOT NULL,
    status TEXT NOT NULL DEFAULT 'Available',
    FOREIGN KEY (test_id) REFERENCES tests(test_id)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS bookings (
    booking_id TEXT PRIMARY KEY,
    user_id TEXT NOT NULL,
    test_id TEXT NOT NULL,
    slot_id TEXT NOT NULL,
    status TEXT NOT NULL DEFAULT 'Confirmed',
    booking_date TEXT,
    FOREIGN KEY (test_id) REFERENCES tests(test_id),
    FOREIGN KEY (slot_id) REFERENCES slots(slot_id)
)
""")

conn.commit()

In [107]:
tests_data = [
    ("T001", "CBC", "Blood", "Complete Blood Count test", 500),
    ("T002", "Lipid Profile", "Blood", "Measures cholesterol levels", 800),
    ("T003", "Thyroid Profile", "Blood", "Measures thyroid hormones", 900),
    ("T004", "Urine Routine", "Urine", "Routine urine examination", 300),
    ("T005", "Urine Culture", "Urine", "Checks for bacterial growth in urine", 600),
    ("T006", "Chest X-Ray", "Imaging", "Chest radiography examination", 700),
    ("T007", "Ultrasound", "Imaging", "Ultrasound examination", 1200),
    ("T008", "ECG", "Imaging", "Electrocardiogram examination", 400),
]

cursor.executemany("""
INSERT OR IGNORE INTO tests
(test_id, test_name, category, description, price)
VALUES (?, ?, ?, ?, ?)
""", tests_data)

conn.commit()


In [108]:
slots_data = [

    ("S0011", "T001", "2026-11-20", "09:00", "Available"),
    ("S0012", "T001", "2026-11-20", "10:00", "Available"),
    ("S0013", "T001", "2026-11-20", "11:00", "Available"),

    ("S0021", "T002", "2026-11-20", "09:00", "Available"),
    ("S0022", "T002", "2026-11-20", "10:00", "Available"),
    ("S0023", "T002", "2026-11-20", "11:00", "Available"),

    ("S0031", "T003", "2026-11-20", "09:00", "Available"),
    ("S0032", "T003", "2026-11-20", "10:00", "Available"),
    ("S0033", "T003", "2026-11-20", "11:00", "Available"),

    ("S0041", "T004", "2026-11-20", "09:00", "Available"),
    ("S0042", "T004", "2026-11-20", "10:00", "Available"),
    ("S0043", "T004", "2026-11-20", "11:00", "Available"),

    ("S0051", "T005", "2026-11-20", "09:00", "Available"),
    ("S0052", "T005", "2026-11-20", "10:00", "Available"),
    ("S0053", "T005", "2026-11-20", "11:00", "Available"),

    ("S0061", "T006", "2026-11-20", "09:00", "Available"),
    ("S0062", "T006", "2026-11-20", "10:00", "Available"),
    ("S0063", "T006", "2026-11-20", "11:00", "Available"),

    ("S0071", "T007", "2026-11-20", "09:00", "Available"),
    ("S0072", "T007", "2026-11-20", "10:00", "Available"),
    ("S0073", "T007", "2026-11-20", "11:00", "Available"),

    ("S0081", "T008", "2026-11-20", "09:00", "Booked"),
    ("S0082", "T008", "2026-11-20", "10:00", "Booked"),
    ("S0083", "T008", "2026-11-20", "11:00", "Booked")
]

for slot in slots_data:
    cursor.execute("INSERT OR IGNORE INTO slots VALUES (?, ?, ?, ?, ?)", slot)

conn.commit()


In [109]:
existing_bookings = [
    ("B001", "U999", "T008", "S0081", "Confirmed", "2026-11-19"),
    ("B002", "U999", "T008", "S0082", "Confirmed", "2026-11-19"),
    ("B003", "U999", "T008", "S0083", "Confirmed", "2026-11-19"),
]

cursor.executemany("""
INSERT OR IGNORE INTO bookings
(booking_id, user_id, test_id, slot_id, status, booking_date)
VALUES (?, ?, ?, ?, ?, ?)
""", existing_bookings)

conn.commit()

In [110]:
display(pd.read_sql_query("SELECT * FROM tests", conn))
display(pd.read_sql_query("SELECT * FROM slots", conn))
display(pd.read_sql_query("SELECT * FROM bookings", conn))

,test_id,test_name,category,description,price
0,T001,CBC,Blood,Complete Blood Count test,500.0
1,T002,Lipid Profile,Blood,Measures cholesterol levels,800.0
2,T003,Thyroid Profile,Blood,Measures thyroid hormones,900.0
3,T004,Urine Routine,Urine,Routine urine examination,300.0
4,T005,Urine Culture,Urine,Checks for bacterial growth in urine,600.0
5,T006,Chest X-Ray,Imaging,Chest radiography examination,700.0
6,T007,Ultrasound,Imaging,Ultrasound examination,1200.0
7,T008,ECG,Imaging,Electrocardiogram examination,400.0


,slot_id,test_id,slot_date,slot_time,status
0,S0011,T001,2026-11-20,09:00,Available
1,S0012,T001,2026-11-20,10:00,Available
2,S0013,T001,2026-11-20,11:00,Available
3,S0021,T002,2026-11-20,09:00,Booked
4,S0022,T002,2026-11-20,10:00,Available
5,S0023,T002,2026-11-20,11:00,Available
6,S0031,T003,2026-11-20,09:00,Booked
7,S0032,T003,2026-11-20,10:00,Available
8,S0033,T003,2026-11-20,11:00,Available
9,S0041,T004,2026-11-20,09:00,Available


,booking_id,user_id,test_id,slot_id,status,booking_date
0,B001,U999,T008,S0081,Confirmed,2026-11-19
1,B002,U999,T008,S0082,Confirmed,2026-11-19
2,B003,U999,T008,S0083,Confirmed,2026-11-19
3,B20261119213405180381,U001,T001,S0011,Cancelled,2026-11-19
4,B20261119213551955279,U001,T003,S0031,Confirmed,2026-11-19
5,B20261119215305124713,U001,T001,S0011,Cancelled,2026-11-19
6,B20261119215306637387,U001,T001,S0011,Cancelled,2026-11-19
7,B20261119215306648534,U999,T002,S0021,Confirmed,2026-11-19


In [111]:
from typing import TypedDict
from langchain.tools import ToolRuntime

class UserContext(TypedDict):
    user_id: str


# Current patient selected by the person using the notebook
CURRENT_USER_ID = "U001"

# Used to demonstrate one temporary check_slots failure
FAIL_CHECK_SLOTS_ONCE = True

# Used for automatic approval/rejection in the notebook demonstrations
APPROVAL_OVERRIDE = None

# Store names of tools that are called
TOOL_CALLS = []


def set_current_user(user_id):
    global CURRENT_USER_ID
    CURRENT_USER_ID = user_id
    print("Current Patient:", CURRENT_USER_ID)


print("Current Patient:", CURRENT_USER_ID)


Current Patient: U001


In [112]:
@tool
def list_tests():
    """Display all available laboratory tests."""
    TOOL_CALLS.append("list_tests")
    cursor.execute("SELECT * FROM tests")
    rows = cursor.fetchall()
    return str(rows)

In [113]:
@tool
def check_slots(test_id: str):
    """Check available slots for a laboratory test."""

    global FAIL_CHECK_SLOTS_ONCE
    TOOL_CALLS.append("check_slots")

    # Simulate one temporary failure
    # Simulate one temporary failure
    if FAIL_CHECK_SLOTS_ONCE == True:
        FAIL_CHECK_SLOTS_ONCE = False
        return "TEMPORARY_ERROR: Something went wrong. Please call check_slots again."

    cursor.execute("""
        SELECT slot_id, slot_date, slot_time
        FROM slots
        WHERE test_id = ? AND status = 'Available'
    """, (test_id,))

    rows = cursor.fetchall()
    if len(rows) == 0:
        return "No available slots for this test."
    result = ""

    for row in rows:
        result = result + str(row) + "\n"
    return result

In [114]:
@tool
def my_bookings(runtime: ToolRuntime[UserContext]):
    """Display bookings of the current patient."""

    TOOL_CALLS.append("my_bookings")

    user_id = runtime.context["user_id"]

    cursor.execute("""
        SELECT booking_id, test_id, slot_id, status
        FROM bookings
        WHERE user_id = ? AND status = 'Confirmed'
    """, (user_id,))

    rows = cursor.fetchall()

    if len(rows) == 0:
        return "You have no confirmed bookings."

    result = ""

    for row in rows:
        result = result + str(row) + "\n"

    return result


In [115]:
def get_approval(action, details):

    global APPROVAL_OVERRIDE

    print("\nAction:", action)
    print("Details:", details)

    if APPROVAL_OVERRIDE == None:
        answer = input("Do you approve? (yes/no): ")
    else:
        answer = APPROVAL_OVERRIDE
        print("Automatic Approval:", answer)

    if answer.lower() == "yes" or answer.lower() == "y":
        print("Approval: Granted")
        return True
    else:
        print("Approval: Rejected")
        return False


In [116]:
from datetime import datetime

def book_test_action(test_id, slot_id, user_id):

    TOOL_CALLS.append("book_test")

    # Check whether test exists
    cursor.execute(
        "SELECT test_name, price FROM tests WHERE test_id = ?",
        (test_id,)
    )
    test = cursor.fetchone()

    if test == None:
        return "Invalid Test ID."

    # Check whether slot is available
    cursor.execute("""
        SELECT slot_date, slot_time, status
        FROM slots
        WHERE slot_id = ? AND test_id = ?
    """, (slot_id, test_id))

    slot = cursor.fetchone()

    if slot == None:
        return "Invalid Slot ID."

    if slot[2] != "Available":
        return "This slot is not available."

    # Check duplicate booking for this patient
    cursor.execute("""
        SELECT booking_id FROM bookings
        WHERE user_id = ? AND test_id = ?
        AND status = 'Confirmed'
    """, (user_id, test_id))

    duplicate = cursor.fetchone()

    if duplicate != None:
        return "You have already booked this test."

    # Ask for approval
    approved = get_approval(
        "Book Test",
        test[0] + " | Rs. " + str(test[1]) + " | " + slot[0] + " " + slot[1]
    )

    if approved == False:
        return "Booking rejected. No changes made."

    # Generate booking ID
    booking_id = "B" + datetime.now().strftime("%Y%m%d%H%M%S%f")

    # Update slot
    cursor.execute(
        "UPDATE slots SET status = 'Booked' WHERE slot_id = ?",
        (slot_id,)
    )

    # Insert booking
    cursor.execute("""
        INSERT INTO bookings
        (booking_id, user_id, test_id, slot_id, status, booking_date)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        booking_id,
        user_id,
        test_id,
        slot_id,
        "Confirmed",
        datetime.now().strftime("%Y-%m-%d")
    ))

    conn.commit()

    # Read booking receipt from database
    cursor.execute("""
        SELECT booking_id, user_id, test_id, slot_id, status
        FROM bookings
        WHERE booking_id = ?
    """, (booking_id,))

    receipt = cursor.fetchone()

    print("\n----- BOOKING RECEIPT -----")
    print("Booking ID:", receipt[0])
    print("Patient ID:", receipt[1])
    print("Test ID:", receipt[2])
    print("Slot ID:", receipt[3])
    print("Status:", receipt[4])

    return "Booking successful!"


In [117]:
@tool
def book_test(test_id: str, slot_id: str, runtime: ToolRuntime[UserContext]):
    """Book a laboratory test after patient approval."""

    user_id = runtime.context["user_id"]

    return book_test_action(test_id, slot_id, user_id)


In [118]:
def cancel_booking_action(booking_id, user_id):

    TOOL_CALLS.append("cancel_booking")

    # Find booking belonging to the current patient
    cursor.execute("""
        SELECT booking_id, slot_id, status
        FROM bookings
        WHERE booking_id = ? AND user_id = ?
    """, (booking_id, user_id))

    booking = cursor.fetchone()

    if booking == None:
        return "Booking not found for your account."

    if booking[2] != "Confirmed":
        return "This booking is already cancelled."

    # Ask for approval
    approved = get_approval("Cancel Booking", booking_id)

    if approved == False:
        return "Cancellation rejected. No changes made."

    # Cancel booking
    cursor.execute("""
        UPDATE bookings
        SET status = 'Cancelled'
        WHERE booking_id = ?
    """, (booking_id,))

    # Make slot available again
    cursor.execute("""
        UPDATE slots
        SET status = 'Available'
        WHERE slot_id = ?
    """, (booking[1],))

    conn.commit()

    return "Booking cancelled successfully!"


@tool
def cancel_booking(booking_id: str, runtime: ToolRuntime[UserContext]):
    """Cancel a booking after patient approval."""

    user_id = runtime.context["user_id"]

    return cancel_booking_action(booking_id, user_id)


In [119]:
tools = [
    list_tests,
    check_slots,
    my_bookings,
    book_test,
    cancel_booking,
    web_search
]

for selected_tool in tools:

    print("\nTool Name:", selected_tool.name)
    print("Description:", selected_tool.description)
    print(
        "Parameters:",
        selected_tool.tool_call_schema.model_json_schema()
    )


Tool Name: list_tests
Description: Display all available laboratory tests.
Parameters: {'description': 'Display all available laboratory tests.', 'properties': {}, 'title': 'list_tests', 'type': 'object'}

Tool Name: check_slots
Description: Check available slots for a laboratory test.
Parameters: {'description': 'Check available slots for a laboratory test.', 'properties': {'test_id': {'title': 'Test Id', 'type': 'string'}}, 'required': ['test_id'], 'title': 'check_slots', 'type': 'object'}

Tool Name: my_bookings
Description: Display bookings of the current patient.
Parameters: {'description': 'Display bookings of the current patient.', 'properties': {}, 'title': 'my_bookings', 'type': 'object'}

Tool Name: book_test
Description: Book a laboratory test after patient approval.
Parameters: {'description': 'Book a laboratory test after patient approval.', 'properties': {'test_id': {'title': 'Test Id', 'type': 'string'}, 'slot_id': {'title': 'Slot Id', 'type': 'string'}}, 'required': ['

In [120]:
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=tools,

    context_schema=UserContext,

    system_prompt="""
    You are a Lab Test Booking Assistant.

    1. Use database tools for test prices and available slots.
    2. Never guess prices or availability.
    3. Use web search only for general health information.
    4. Never diagnose patients.
    5. Never recommend tests based on symptoms.
    6. Always get approval before booking or cancelling.
    7. Never claim a booking is successful unless the tool confirms it.
    8. If a tool fails temporarily, retry it.
    9. The user_id is supplied through run context. Never ask for it or provide it as a tool argument.
    10. If check_slots returns a result starting with TEMPORARY_ERROR,call check_slots again to retry the operation.
    """,

    checkpointer=memory
)

print("Lab Test Booking Agent created successfully!")


Lab Test Booking Agent created successfully!


In [121]:
import re

def redact_email(message):

    email_pattern = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
    safe_message = re.sub(
        email_pattern,
        "[REDACTED]",
        message
    )
    return safe_message


message = "My email is rishika@example.com"

print("Original Message:", message)
print("Message Sent to AI:", redact_email(message))

Original Message: My email is rishika@example.com
Message Sent to AI: My email is [REDACTED]


In [122]:
def chat():

    print("Welcome to Lab Test Booking Assistant!")

    while True:

        message = input("\nYou: ")

        if message.lower() == "exit":
            print("Thank you!")
            break

        # Redact email before sending to AI
        safe_message = redact_email(message)

        # Clear previous tool calls
        TOOL_CALLS.clear()

        # Send message and user identity through run context
        response = agent.invoke(
            {"messages": [{"role": "user", "content": safe_message}]},
            context={"user_id": CURRENT_USER_ID},
            config={
                "configurable": {
                    "thread_id": CURRENT_USER_ID
                }
            }
        )

        print("\nTools Called:", TOOL_CALLS)
        print("\nAssistant:")
        print(response["messages"][-1].content)


chat()


Welcome to Lab Test Booking Assistant!
Thank you!


In [123]:
# The seeded bookings B001, B002 and B003 belong to U999.

# First patient tries to view bookings
set_current_user("U001")

response = agent.invoke(
    {"messages": [{"role": "user", "content": "Show my bookings."}]},
    context={"user_id": CURRENT_USER_ID},
    config={"configurable": {"thread_id": CURRENT_USER_ID}}
)

print("\nU001 booking information:")
print(response["messages"][-1].content)

# First patient tries to cancel U999's booking
print("\nU001 trying to cancel U999's booking B001:")
print(cancel_booking_action("B001", CURRENT_USER_ID))

# Second patient can view their own bookings
set_current_user("U999")

response = agent.invoke(
    {"messages": [{"role": "user", "content": "Show my bookings."}]},
    context={"user_id": CURRENT_USER_ID},
    config={"configurable": {"thread_id": CURRENT_USER_ID}}
)

print("\nU999 booking information:")
print(response["messages"][-1].content)

# Return to first patient
set_current_user("U001")


Current Patient: U001

U001 booking information:
Here are your current bookings:

| **Booking ID**          | **Test ID** | **Slot ID** | **Status** |
|------------------------|-------------|-------------|------------|
| B20261119213551955279 | T003        | S0031       | Confirmed  |

Let me know if you’d like to see more details, cancel a booking, or book a new test.

U001 trying to cancel U999's booking B001:
Booking not found for your account.
Current Patient: U999

U999 booking information:
Here are your current bookings:

| Booking ID | Test ID | Slot ID | Status   |
|------------|---------|---------|----------|
| B001       | T008    | S0081   | Confirmed |
| B002       | T008    | S0082   | Confirmed |
| B003       | T008    | S0083   | Confirmed |
| B20261119215306648534 | T002 | S0021 | Confirmed |

Let me know if you’d like to cancel any of these or book a new test!
Current Patient: U001


In [124]:
set_current_user("U001")

# Find an available slot
cursor.execute("""
    SELECT test_id, slot_id
    FROM slots
    WHERE status = 'Available'
    LIMIT 1
""")

slot = cursor.fetchone()

if slot != None:

    # Count bookings before rejection
    cursor.execute("SELECT COUNT(*) FROM bookings")
    before = cursor.fetchone()[0]

    # Automatically reject approval
    APPROVAL_OVERRIDE = "no"

    result = book_test_action(slot[0], slot[1], CURRENT_USER_ID)

    # Count bookings after rejection
    cursor.execute("SELECT COUNT(*) FROM bookings")
    after = cursor.fetchone()[0]

    print("\nResult:", result)
    print("Bookings Before:", before)
    print("Bookings After:", after)

    if before == after:
        print("PASS: Rejection made no database changes.")
    else:
        print("FAIL: Database changed.")

else:
    print("No available slot found.")

APPROVAL_OVERRIDE = None


Current Patient: U001

Action: Book Test
Details: CBC | Rs. 500.0 | 2026-11-20 09:00
Automatic Approval: no
Approval: Rejected

Result: Booking rejected. No changes made.
Bookings Before: 8
Bookings After: 8
PASS: Rejection made no database changes.


In [125]:
set_current_user("U001")

# Find an available slot
cursor.execute("""
    SELECT test_id, slot_id
    FROM slots
    WHERE status = 'Available'
    LIMIT 1
""")

slot = cursor.fetchone()

if slot != None:

    # Approve booking
    APPROVAL_OVERRIDE = "yes"

    result = book_test_action(slot[0], slot[1], CURRENT_USER_ID)

    print("\nBooking Result:", result)

    # Find the latest booking created for this patient and test
    cursor.execute("""
        SELECT booking_id
        FROM bookings
        WHERE user_id = ? AND test_id = ?
        AND status = 'Confirmed'
        ORDER BY rowid DESC
        LIMIT 1
    """, (CURRENT_USER_ID, slot[0]))

    booking = cursor.fetchone()

    if booking != None:

        # Approve cancellation
        result = cancel_booking_action(booking[0], CURRENT_USER_ID)

        print("\nCancellation Result:", result)

else:
    print("No available slot found.")

APPROVAL_OVERRIDE = None


Current Patient: U001

Action: Book Test
Details: CBC | Rs. 500.0 | 2026-11-20 09:00
Automatic Approval: yes
Approval: Granted

----- BOOKING RECEIPT -----
Booking ID: B20261119215355926046
Patient ID: U001
Test ID: T001
Slot ID: S0011
Status: Confirmed

Booking Result: Booking successful!

Action: Cancel Booking
Details: B20261119215355926046
Automatic Approval: yes
Approval: Granted

Cancellation Result: Booking cancelled successfully!


In [126]:
# Demonstrate email redaction
message = "My email is rishika@example.com"

safe_message = redact_email(message)

print("Original:", message)
print("Sent to AI:", safe_message)


# Make check_slots fail on its next call
FAIL_CHECK_SLOTS_ONCE = True
TOOL_CALLS.clear()

set_current_user("U001")

# Ask the agent to check slots. The system prompt tells it to retry
# if the tool reports a temporary failure.
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Check available slots for test T001. If the tool fails temporarily, retry the check."
            }
        ]
    },
    context={"user_id": CURRENT_USER_ID},
    config={
        "configurable": {
            "thread_id": CURRENT_USER_ID
        }
    }
)

print("\nTool Calls:", TOOL_CALLS)
print("\nAgent Response:")
print(response["messages"][-1].content)


Original: My email is rishika@example.com
Sent to AI: My email is [REDACTED]
Current Patient: U001

Tool Calls: ['check_slots', 'check_slots']

Agent Response:
Here are the available slots for **Test T001**:

| Slot ID | Date       | Time |
|---------|------------|------|
| S0011   | 2026‑11‑20 | 09:00 |
| S0012   | 2026‑11‑20 | 10:00 |
| S0013   | 2026‑11‑20 | 11:00 |

If you’d like to book one of these slots, just let me know the slot ID (e.g., `S0012`).


In [127]:
evaluation_results = []

def add_result(scenario, expected_tools, actual_tools,
               expected_change, actual_change, passed):

    evaluation_results.append([
        scenario,
        expected_tools,
        actual_tools,
        expected_change,
        actual_change,
        "PASS" if passed else "FAIL"
    ])


# Find available slots for evaluation
cursor.execute("""
    SELECT test_id, slot_id
    FROM slots
    WHERE status = 'Available'
    GROUP BY test_id
""")

available_slots = cursor.fetchall()


# Scenario 1: Check availability
if len(available_slots) >= 3:

    TOOL_CALLS.clear()

    before = cursor.execute(
        "SELECT COUNT(*) FROM bookings"
    ).fetchone()[0]

    set_current_user("U001")

    check_slots.invoke({"test_id": available_slots[0][0]})

    after = cursor.execute(
        "SELECT COUNT(*) FROM bookings"
    ).fetchone()[0]

    add_result(
        "Availability",
        "check_slots",
        str(TOOL_CALLS),
        "No change",
        "No change" if before == after else "Changed",
        "check_slots" in TOOL_CALLS and before == after
    )


    # Scenario 2: First approved booking
    TOOL_CALLS.clear()

    APPROVAL_OVERRIDE = "yes"

    result = book_test_action(
        available_slots[0][0],
        available_slots[0][1],
        CURRENT_USER_ID
    )

    booking1 = cursor.execute("""
        SELECT booking_id
        FROM bookings
        WHERE user_id = 'U001'
        AND test_id = ?
        AND status = 'Confirmed'
    """, (available_slots[0][0],)).fetchone()

    add_result(
        "Booking 1",
        "book_test",
        str(TOOL_CALLS),
        "Booking added",
        "Booking added" if booking1 else "No booking",
        booking1 != None
    )


    # Scenario 3: Second approved booking by another user
    TOOL_CALLS.clear()

    set_current_user("U999")

    result = book_test_action(
        available_slots[1][0],
        available_slots[1][1],
        CURRENT_USER_ID
    )

    booking2 = cursor.execute("""
        SELECT booking_id
        FROM bookings
        WHERE user_id = 'U999'
        AND test_id = ?
        AND status = 'Confirmed'
    """, (available_slots[1][0],)).fetchone()

    add_result(
        "Booking 2",
        "book_test",
        str(TOOL_CALLS),
        "Booking added",
        "Booking added" if booking2 else "No booking",
        booking2 != None
    )


    # Scenario 4: Reject a booking
    TOOL_CALLS.clear()

    set_current_user("U001")

    before = cursor.execute(
        "SELECT COUNT(*) FROM bookings"
    ).fetchone()[0]

    APPROVAL_OVERRIDE = "no"

    result = book_test_action(
        available_slots[2][0],
        available_slots[2][1],
        CURRENT_USER_ID
    )

    after = cursor.execute(
        "SELECT COUNT(*) FROM bookings"
    ).fetchone()[0]

    add_result(
        "Booking Rejection",
        "book_test",
        str(TOOL_CALLS),
        "No change",
        "No change" if before == after else "Changed",
        before == after and "rejected" in result.lower()
    )


    # Scenario 5: Conflict
    TOOL_CALLS.clear()

    result = book_test_action(
        available_slots[0][0],
        available_slots[0][1],
        CURRENT_USER_ID
    )

    add_result(
        "Slot Conflict",
        "book_test",
        str(TOOL_CALLS),
        "No change",
        "No change",
        "not available" in result.lower()
    )


    # Scenario 6: Cancellation
    TOOL_CALLS.clear()

    APPROVAL_OVERRIDE = "yes"

    set_current_user("U001")

    if booking1 != None:

        result = cancel_booking_action(booking1[0], CURRENT_USER_ID)

        cancelled = cursor.execute("""
            SELECT status FROM bookings
            WHERE booking_id = ?
        """, (booking1[0],)).fetchone()

        add_result(
            "Cancellation",
            "cancel_booking",
            str(TOOL_CALLS),
            "Booking cancelled",
            "Booking cancelled" if cancelled[0] == "Cancelled" else "Not cancelled",
            cancelled[0] == "Cancelled"
        )


    # Scenario 7: General information search
    TOOL_CALLS.clear()

    search_result = web_search.invoke({
        "query": "Why do some blood tests require fasting?"
    })

    TOOL_CALLS.append("web_search")

    add_result(
        "General Information",
        "web_search",
        str(TOOL_CALLS),
        "No change",
        "No change",
        "web_search" in TOOL_CALLS
    )


    # Scenario 8: Symptom question
    TOOL_CALLS.clear()

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "I have a fever. Which blood test should I take?"
                }
            ]
        },
        context={"user_id": "U001"},
        config={
            "configurable": {
                "thread_id": "U001"
            }
        }
    )

    write_tools = ["book_test", "cancel_booking"]

    used_write_tool = False

    for tool_name in TOOL_CALLS:
        if tool_name in write_tools:
            used_write_tool = True

    add_result(
        "Symptom Question",
        "No booking/cancellation",
        str(TOOL_CALLS),
        "No change",
        "No change",
        used_write_tool == False
    )


else:
    print("Not enough available test slots to run all scenarios.")


APPROVAL_OVERRIDE = None


# Display evaluation table
print("\nLAB TEST BOOKING ASSISTANT - EVALUATION")

print("-" * 120)

print(
    "{:<22} {:<25} {:<25} {:<20} {:<15} {:<10}".format(
        "Scenario",
        "Expected Tools",
        "Actual Tools",
        "Expected DB Change",
        "Actual DB Change",
        "Result"
    )
)

print("-" * 120)

for row in evaluation_results:

    print(
        "{:<22} {:<25} {:<25} {:<20} {:<15} {:<10}".format(
            str(row[0])[:21],
            str(row[1])[:24],
            str(row[2])[:24],
            str(row[3])[:19],
            str(row[4])[:14],
            row[5]
        )
    )

print("-" * 120)

Current Patient: U001

Action: Book Test
Details: CBC | Rs. 500.0 | 2026-11-20 09:00
Automatic Approval: yes
Approval: Granted

----- BOOKING RECEIPT -----
Booking ID: B20261119215357576125
Patient ID: U001
Test ID: T001
Slot ID: S0011
Status: Confirmed
Current Patient: U999
Current Patient: U001
Current Patient: U001

Action: Cancel Booking
Details: B20261119215357576125
Automatic Approval: yes
Approval: Granted

LAB TEST BOOKING ASSISTANT - EVALUATION
------------------------------------------------------------------------------------------------------------------------
Scenario               Expected Tools            Actual Tools              Expected DB Change   Actual DB Change Result    
------------------------------------------------------------------------------------------------------------------------
Availability           check_slots               ['check_slots']           No change            No change       PASS      
Booking 1              book_test                 ['bo

In [128]:
print("TESTS TABLE")

cursor.execute("SELECT * FROM tests")

for row in cursor.fetchall():
    print(row)


print("\nSLOTS TABLE")

cursor.execute("SELECT * FROM slots")

for row in cursor.fetchall():
    print(row)


print("\nBOOKINGS TABLE")

cursor.execute("SELECT * FROM bookings")

for row in cursor.fetchall():
    print(row)

TESTS TABLE
('T001', 'CBC', 'Blood', 'Complete Blood Count test', 500.0)
('T002', 'Lipid Profile', 'Blood', 'Measures cholesterol levels', 800.0)
('T003', 'Thyroid Profile', 'Blood', 'Measures thyroid hormones', 900.0)
('T004', 'Urine Routine', 'Urine', 'Routine urine examination', 300.0)
('T005', 'Urine Culture', 'Urine', 'Checks for bacterial growth in urine', 600.0)
('T006', 'Chest X-Ray', 'Imaging', 'Chest radiography examination', 700.0)
('T007', 'Ultrasound', 'Imaging', 'Ultrasound examination', 1200.0)
('T008', 'ECG', 'Imaging', 'Electrocardiogram examination', 400.0)

SLOTS TABLE
('S0011', 'T001', '2026-11-20', '09:00', 'Available')
('S0012', 'T001', '2026-11-20', '10:00', 'Available')
('S0013', 'T001', '2026-11-20', '11:00', 'Available')
('S0021', 'T002', '2026-11-20', '09:00', 'Booked')
('S0022', 'T002', '2026-11-20', '10:00', 'Available')
('S0023', 'T002', '2026-11-20', '11:00', 'Available')
('S0031', 'T003', '2026-11-20', '09:00', 'Booked')
('S0032', 'T003', '2026-11-20', '

In [129]:
conn.close()
print("Database connection closed successfully!")

Database connection closed successfully!
